In [1]:
import argparse, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.preprocessing import MinMaxScaler
from utils import download_data, create_features, time_train_test_split, plot_preds, evaluate_and_print

In [2]:
def create_sequences(X, y, seq_len=20):
    Xs, ys = [], []
    for i in range(seq_len, len(X)):
        Xs.append(X[i-seq_len:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

In [3]:
def build_model(input_shape, dropout=0.2):
    model = Sequential()
    model.add(LSTM(64, return_sequences=True, input_shape=input_shape))
    model.add(Dropout(dropout))
    model.add(LSTM(32))
    model.add(Dropout(dropout))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mse')
    return model

Download the data and feratures to use 

In [4]:
ticker='AAPL'
df = download_data(ticker='AAPL', start='2023-01-01', end='2025-01-01', save_csv=None)
df = create_features(df)
features = ['Close','MA5','MA20','Return','Volume']
X = df[features].values
y = df['Target'].values.reshape(-1,1)

print(f'Data features X.shape: {X.shape}')
print(f'Data labels y.shape: {y.shape}')

/Users/jessehernandez/ML_Learning/Projects/stock-price-predictor/src/utils.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start, end=end, progress=False)


Data features X.shape: (482, 5)
Data labels y.shape: (482, 1)


Build the LSTM model

In [5]:
test_size = 0.2
batch_size = 32
epochs = 30
seq_len = 20 #args.seq_len

# scale the features
scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()
X_scaled = scaler_x.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

X_seq, y_seq = create_sequences(X_scaled, y_scaled, seq_len=seq_len)

split = int((1-test_size) * len(X_seq))
X_train, X_test = X_seq[:split], X_seq[split:]
y_train, y_test = y_seq[:split], y_seq[split:]

model = build_model((X_train.shape[1], X_train.shape[2]))
os.makedirs('models', exist_ok=True)
checkpoint = ModelCheckpoint(f'models/lstm_{ticker}.keras', save_best_only=True, monitor='val_loss')
early = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size,
                    validation_split=0.1, callbacks=[checkpoint, early], verbose=1)

# predict and inverse transform
preds_scaled = model.predict(X_test)
preds = scaler_y.inverse_transform(preds_scaled)
actual = scaler_y.inverse_transform(y_test)

os.makedirs('figs', exist_ok=True)
plot_preds(actual, preds, title=f'{ticker} LSTM Predictions', savepath=f'figs/test_lstm_{ticker}.png')

mae = evaluate_and_print(actual, preds)
print(f'LSTM MAE: {mae:.4f}')

2026-01-24 19:56:01.070005: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2026-01-24 19:56:01.070059: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-01-24 19:56:01.070083: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2026-01-24 19:56:01.070509: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-01-24 19:56:01.070532: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
/Users/jessehernandez/miniconda3/envs/tf_build/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. Wh

Epoch 1/30


2026-01-24 19:56:01.743808: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


11/11 ━━━━━━━━━━━━━━━━━━━━ 3s 151ms/step - loss: 0.0122 - val_loss: 0.0557
Epoch 2/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0068 - val_loss: 0.0254
Epoch 3/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0066 - val_loss: 0.0091
Epoch 4/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - loss: 0.0052 - val_loss: 0.0116
Epoch 5/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0046 - val_loss: 0.0135
Epoch 6/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0040 - val_loss: 0.0162
Epoch 7/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0036 - val_loss: 0.0059
Epoch 8/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0043 - val_loss: 0.0061
Epoch 9/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0039 - val_loss: 0.0097
Epoch 10/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0034 - val_loss: 0.0101
Epoch 11/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0031 - val_loss: 0.0069
Epoch 12/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.0036 - val_